In [1]:
# 08_selection_covariate_adjusted.ipynb
# Selection on observables and covariate-adjusted estimates. We document selection
# into baseline private insurance (a propensity model and standardized mean
# differences) and re-estimate the overall shock effect and the insurance buffer
# with a covariate-adjusted, doubly-robust Callaway-Sant'Anna estimator (conditional
# parallel trends). Figure is grayscale, no titles, 600-dpi PNG and PDF; tables to
# ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from differences import ATTgt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
p["che25f"] = p["che25"].astype(float); p["che40f"] = p["che40"].astype(float)

# ---- time-invariant baseline covariates (first observed year) ----
by = p.groupby("PIDWON")["year"].transform("min")
base = p[p["year"] == by].groupby("PIDWON").agg(
    base_age=("age", "first"), base_female=("female", "first"), base_chronic=("chronic", "first"),
    base_disabled=("disabled", "first"), base_loginc=("log_hhinc", "first"),
    base_hhsize=("hh_size", "first")).reset_index()
p = p.merge(base, on="PIDWON", how="left")
covs = ["base_age", "base_female", "base_chronic", "base_disabled", "base_loginc", "base_hhsize"]

def cs(outcome, adj=False, split=None):
    cols = ["PIDWON", "year", "cohort", outcome] + covs + (["base_phi"] if split else [])
    d = p[cols].dropna().copy(); d["cohort"] = d["cohort"].replace(0, np.nan)
    d = d.set_index(["PIDWON", "year"])
    f = outcome + (" ~ " + " + ".join(covs) if adj else "")
    a = ATTgt(data=d, cohort_column="cohort", base_period="universal")
    a.fit(formula=f, est_method="dr", control_group="not_yet_treated", sample_split_column=split, progress_bar=False)
    s = a.aggregate("simple"); s.columns = [str(c) for c in s.columns]
    return s

# ---- overall ATT and buffer: unadjusted vs covariate-adjusted ----
orows, brows = [], []
for th, lab in [("che25f", "CHE 25%"), ("che40f", "CHE 40%")]:
    u = cs(th, False); c = cs(th, True)
    orows.append({"outcome": lab, "unadj_att": float(u.iloc[0, 0]), "unadj_se": float(u.iloc[0, 1]),
                  "adj_att": float(c.iloc[0, 0]), "adj_se": float(c.iloc[0, 1])})
    for adj in (False, True):
        s = cs(th, adj, split="base_phi")
        brows.append({"outcome": lab, "adjusted": adj, "att_nophi": float(s.iloc[0, 0]),
                      "att_phi": float(s.iloc[1, 0]), "buffer": float(s.iloc[1, 0]) - float(s.iloc[0, 0])})
pd.DataFrame(orows).to_csv(f"{TAB}/t08_covadj_overall.csv", index=False)
pd.DataFrame(brows).to_csv(f"{TAB}/t08_covadj_buffer.csv", index=False)

# ---- propensity model and standardized mean differences ----
pb = base.merge(p[["PIDWON", "base_phi"]].drop_duplicates("PIDWON"), on="PIDWON", how="left").dropna(subset=covs + ["base_phi"])
X = StandardScaler().fit_transform(pb[covs]); yv = pb["base_phi"].astype(int)
auc = roc_auc_score(yv, LogisticRegression(max_iter=1000).fit(X, yv).predict_proba(X)[:, 1])
smd = []
for c in covs:
    t = pb[c][yv == 1]; k = pb[c][yv == 0]
    smd.append({"covariate": c, "smd": (t.mean() - k.mean()) / np.sqrt((t.var() + k.var()) / 2)})
bal = pd.DataFrame(smd); bal["prop_auc"] = auc
bal.to_csv(f"{TAB}/t08_balance.csv", index=False)

# ---- figure: standardized mean differences (love plot, grayscale) ----
labmap = {"base_age": "Age", "base_female": "Female", "base_chronic": "Chronic disease",
          "base_disabled": "Disabled", "base_loginc": "Log income", "base_hhsize": "Household size"}
bb = bal.reindex(bal["smd"].abs().sort_values().index)
fig, ax = plt.subplots(figsize=(6.4, 4))
ax.scatter(bb["smd"], range(len(bb)), color="0.15", zorder=3, s=45)
ax.axvline(0, color="0.5", lw=0.8); ax.axvline(0.1, color="0.7", ls="--", lw=1); ax.axvline(-0.1, color="0.7", ls="--", lw=1)
ax.set_yticks(range(len(bb))); ax.set_yticklabels([labmap[c] for c in bb["covariate"]])
ax.set_xlabel("Standardized mean difference (insured $-$ uninsured)")
save_fig(fig, "fig11_selection_balance")

print("Overall ATT (unadj vs covariate-adjusted):")
print(pd.DataFrame(orows).round(4).to_string(index=False))
print("\nBuffer (unadj vs adjusted):")
print(pd.DataFrame(brows).round(4).to_string(index=False))
print(f"\nPropensity AUC for baseline PHI = {auc:.3f}")


Overall ATT (unadj vs covariate-adjusted):
outcome  unadj_att  unadj_se  adj_att  adj_se
CHE 25%     0.0683    0.0043   0.0706  0.0045
CHE 40%     0.0357    0.0030   0.0368  0.0031

Buffer (unadj vs adjusted):
outcome  adjusted  att_nophi  att_phi  buffer
CHE 25%     False     0.1246   0.0532 -0.0715
CHE 25%      True     0.1290   0.0541 -0.0749
CHE 40%     False     0.0727   0.0257 -0.0471
CHE 40%      True     0.0732   0.0260 -0.0472

Propensity AUC for baseline PHI = 0.795
